# 🧼 Handwash AI — Full Training Pipeline
**Phiên bản:** LOSO (Leave-One-Subject-Out) Cross-Validation

---
## Cấu trúc thư mục Google Drive:
```
Webcam_Handwash/
├── bao_data/          ← Dữ liệu mới (BAO1.mov–BAO14.mov + label)
└── kaggle_data/
    ├── kaggle-dataset-6classes/   ← Video thô Kaggle
    └── processed_features/        ← Features đã trích xuất
```
**Ưu tiên:** bao_data được ưu tiên khi có xung đột với kaggle_data

In [ ]:
# ============================================================
# CELL 0: MOUNT GOOGLE DRIVE + KIỂM TRA GPU
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mounted at /content/drive
Mon Jun 29 07:43:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [ ]:
# ============================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & TỰ ĐỘNG RESTART KERNEL
# ============================================================
# Gỡ tensorflow để tránh xung đột protobuf với mediapipe
!pip uninstall -y tensorflow
# Cài đặt numpy < 2.0 và upgrade pandas để tránh lỗi numpy.dtype size changed
!pip install -q "numpy<2.0" pandas --upgrade
!pip install -q mediapipe==0.10.21 tqdm scipy scikit-learn seaborn openpyxl

import os
print('✅ Cài đặt thư viện hoàn tất!')
print('⚠️ ĐANG TỰ ĐỘNG KHỞI ĐỘNG LẠI RUNTIME (RESTART KERNEL)...')
print('⚠️ SAU KHI RESTART XONG, HÃY CHẠY TIẾP TỪ CELL 2 (hoặc CELL 4).')
os.kill(os.getpid(), 9)


Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:


In [ ]:
# ============================================================
# CELL 2: KHÁM PHÁ CẤU TRÚC DỮ LIỆU & ĐỌC LABEL
# ============================================================
import os
import pandas as pd

BASE = '/content/drive/MyDrive/Webcam_Handwash'
BAO_DIR = os.path.join(BASE, 'bao_data')
KAGGLE_DIR = os.path.join(BASE, 'kaggle_data')

print('='*60)
print('📂 BÁO CÁO CẤU TRÚC DỮ LIỆU')
print('='*60)

# --- BAO_DATA ---
print('\n📁 bao_data/')
bao_files = sorted(os.listdir(BAO_DIR))
for f in bao_files:
    fpath = os.path.join(BAO_DIR, f)
    if os.path.isdir(fpath):
        print(f'  📁 {f}/')
    else:
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  📄 {f} ({size_mb:.1f} MB)')

# --- KAGGLE_DATA ---
print('\n📁 kaggle_data/')
if os.path.exists(KAGGLE_DIR):
    for item in sorted(os.listdir(KAGGLE_DIR)):
        item_path = os.path.join(KAGGLE_DIR, item)
        if os.path.isdir(item_path):
            sub_items = os.listdir(item_path)
            print(f'  📁 {item}/ ({len(sub_items)} items)')
            if item == 'processed_features':
                for cls in sorted(sub_items):
                    cls_path = os.path.join(item_path, cls)
                    if os.path.isdir(cls_path):
                        n = len(os.listdir(cls_path))
                        print(f'    Class {cls}: {n} files')

# --- ĐỌC LABEL FILE ---
print('\n' + '='*60)
print('📋 ĐỌC FILE LABEL BAO_DATA')
print('='*60)

label_df = None
# Tìm đệ quy để vào tận trong thư mục 'label' nếu có
for root, dirs, files in os.walk(BAO_DIR):
    for fname in files:
        if fname.endswith('.xlsx') or fname.endswith('.xls') or fname.endswith('.csv'):
            if 'label' in fname.lower() or 'label' in root.lower():
                label_path = os.path.join(root, fname)
                print(f'Tìm thấy label file: {label_path}')
                try:
                    if fname.endswith('.xlsx') or fname.endswith('.xls'):
                        label_df = pd.read_excel(label_path)
                    elif fname.endswith('.csv'):
                        label_df = pd.read_csv(label_path)
                except Exception as e:
                    print(f'Lỗi đọc label: {e}')
                break
    if label_df is not None:
        break

if label_df is not None:
    print(f'\nShape: {label_df.shape}')
    print(f'Columns: {list(label_df.columns)}')
    print('\nNội dung đầy đủ:')
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    print(label_df.to_string())
else:
    print('⚠️ Không tìm thấy file label! Vui lòng kiểm tra lại tên file.')

In [ ]:
# ============================================================
# CELL 3: KIỂM TRA KHI CÓ PROCESSED FEATURES KAGGLE
# Nếu đã có processed_features → dùng ngay, không cần trích xuất lại
# ============================================================
import numpy as np

KAGGLE_PROCESSED = os.path.join(KAGGLE_DIR, 'processed_features')

print('='*60)
print('📊 THỐNG KÊ KAGGLE PROCESSED FEATURES')
print('='*60)

kaggle_stats = {}
if os.path.exists(KAGGLE_PROCESSED):
    classes = sorted([d for d in os.listdir(KAGGLE_PROCESSED)
                      if os.path.isdir(os.path.join(KAGGLE_PROCESSED, d))])
    total = 0
    for cls in classes:
        cls_path = os.path.join(KAGGLE_PROCESSED, cls)
        files = [f for f in os.listdir(cls_path) if f.endswith('.npy')]
        kaggle_stats[cls] = files
        total += len(files)

        # Đọc 1 file để verify
        if files:
            sample = np.load(os.path.join(cls_path, files[0]), allow_pickle=True).item()
            img_shape = sample['img_feat'].shape
            skel_shape = sample['skel_feat'].shape
            print(f'  Class {cls}: {len(files):3d} files | img={img_shape} skel={skel_shape}')
        else:
            print(f'  Class {cls}: 0 files ⚠️')

    print(f'\n  TỔNG: {total} files hợp lệ')
    print(f'  img_feat dim: {img_shape[1]} | skel_feat dim: {skel_shape[1]}')

    # Kiểm tra subject IDs
    subjects = set()
    for cls, files in kaggle_stats.items():
        for f in files:
            parts = f.replace('.npy', '').split('_')
            for i, p in enumerate(parts):
                if p == 'G' and i+1 < len(parts):
                    subjects.add(f'G_{parts[i+1]}')
                    break
    print(f'  Subjects tìm thấy: {sorted(subjects)}')
else:
    print('⚠️ Không tìm thấy processed_features! Cần chạy preprocess từ raw video.')

In [ ]:
# ============================================================
# CELL 4: TRÍCH XUẤT FEATURES TỪ BAO_DATA
# Cắt video theo label → tạo file .npy giống kaggle format
# Output: /content/bao_processed/<class>/<video_step>.npy
# ============================================================
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models
import torchvision.transforms as T
import mediapipe as mp
from tqdm.notebook import tqdm
import os, glob

# --- CONFIG ---
BAO_PROCESSED_OUT = '/content/bao_processed'
SUBJECT_NAME = 'BAO'  # Subject ID cho LOSO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ---- HybridFeatureExtractor ----
class HybridFeatureExtractor:
    def __init__(self, device):
        self.device = device

        # MobileNetV3-Small backbone (giống kaggle pipeline)
        mobilenet = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        self.img_encoder = nn.Sequential(*list(mobilenet.children())[:-1]).to(device)
        self.img_encoder.eval()
        self.img_dim = 576

        # MediaPipe HandLandmarker
        if not os.path.exists('/content/hand_landmarker.task'):
            os.system('curl -sLo /content/hand_landmarker.task '
                      'https://storage.googleapis.com/mediapipe-models/'
                      'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task')

        from mediapipe.tasks import python
        from mediapipe.tasks.python import vision
        base_options = python.BaseOptions(model_asset_path='/content/hand_landmarker.task')
        options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
        self.detector = vision.HandLandmarker.create_from_options(options)

        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def extract_frames(self, cap, start_frame, end_frame):
        """Trích xuất features từ một đoạn video [start_frame, end_frame)"""
        img_features = []
        skel_features = []

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        frame_idx = start_frame

        while frame_idx < end_frame:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            # --- Skeleton ---
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            results = self.detector.detect(mp_image)

            skel_feat = np.zeros((2, 21, 3), dtype=np.float32)
            if results.hand_landmarks and results.handedness:
                for hand_lms, handedness in zip(results.hand_landmarks, results.handedness):
                    label = handedness[0].category_name
                    idx = 0 if label == 'Left' else 1
                    wrist = np.array([hand_lms[0].x, hand_lms[0].y, hand_lms[0].z])
                    for i in range(21):
                        lm = hand_lms[i]
                        skel_feat[idx, i] = np.array([lm.x, lm.y, lm.z]) - wrist
            skel_features.append(skel_feat.flatten())

            # --- Image CNN ---
            img_tensor = self.transform(frame_rgb).unsqueeze(0).to(self.device)
            with torch.no_grad():
                feat = self.img_encoder(img_tensor)  # [1, 576, 1, 1]
                img_features.append(feat.cpu().numpy().flatten())

        return np.array(img_features, dtype=np.float32), np.array(skel_features, dtype=np.float32)


# ---- PARSE LABEL FILE ----
# Đọc label để biết video nào, step nào, frame nào
def parse_label_file(bao_dir):
    label_df = None
    for root, dirs, files in os.walk(bao_dir):
        for fname in sorted(files):
            if fname.endswith('.xlsx') or fname.endswith('.xls') or fname.endswith('.csv'):
                if 'label' in fname.lower() or 'label' in root.lower():
                    label_path = os.path.join(root, fname)
                    print(f'Đọc label: {label_path}')
                    if fname.endswith('.xlsx') or fname.endswith('.xls'):
                        label_df = pd.read_excel(label_path)
                    elif fname.endswith('.csv'):
                        label_df = pd.read_csv(label_path)
                    return label_df
    return label_df


label_df = parse_label_file(BAO_DIR)

if label_df is not None:
    print('\n=== LABEL FILE ===')
    print(f'Columns: {list(label_df.columns)}')
    print(label_df.to_string())
else:
    print('⚠️ Không tìm thấy label file!')

In [ ]:
# ============================================================
# CELL 5: CHẠY TRÍCH XUẤT BAO_DATA → .NPY FILES
# Chạy sau khi xác nhận label_df ở Cell 4 đúng format.
# Tự động detect columns từ label file.
# ============================================================
import pandas as pd
import numpy as np
import os
import cv2
from tqdm.notebook import tqdm

# ---- TỰ ĐỘNG MAP COLUMNS ----
# Phát hiện tên cột tự động
def detect_columns(df):
    """
    Tự động phát hiện tên cột cho: video, step, start, end
    Trả về dict {'video': col_name, 'step': col_name, 'start': col_name, 'end': col_name}
    """
    cols = {c.lower(): c for c in df.columns}
    col_map = {}

    # Video name
    for key in ['video', 'video_name', 'file', 'filename', 'video_id', 'clip']:
        if key in cols:
            col_map['video'] = cols[key]
            break

    # Step / class
    for key in ['step', 'class', 'label', 'action', 'category', 'gesture']:
        if key in cols:
            col_map['step'] = cols[key]
            break

    # Start frame/time
    for key in ['start_frame', 'start', 'begin', 'from', 'start_time', 'begin_frame']:
        if key in cols:
            col_map['start'] = cols[key]
            break

    # End frame/time
    for key in ['end_frame', 'end', 'to', 'finish', 'end_time', 'stop_frame']:
        if key in cols:
            col_map['end'] = cols[key]
            break

    return col_map


# CLASS MAP: Ánh xạ tên step → class index 0-6
# Kaggle format: class 0=Background, 1=Step1...6=Step6
STEP_TO_CLASS = {
    0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6,   # số nguyên
    'step1': 1, 'step2': 2, 'step3': 3, 'step4': 4, 'step5': 5, 'step6': 6,
    'bg': 0, 'background': 0,
    'b1': 1, 'b2': 2, 'b3': 3, 'b4': 4, 'b5': 5, 'b6': 6,
    '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '0': 0,
}

def map_step_to_class(step_val):
    if isinstance(step_val, (int, float)):
        return STEP_TO_CLASS.get(int(step_val), -1)
    return STEP_TO_CLASS.get(str(step_val).lower().strip(), -1)


def extract_bao_data(label_df, bao_dir, out_dir, subject_name='BAO'):
    """Cắt video bao_data theo label → lưu .npy"""
    os.makedirs(out_dir, exist_ok=True)
    for cls in range(7):  # 0-6
        os.makedirs(os.path.join(out_dir, str(cls)), exist_ok=True)

    extractor = HybridFeatureExtractor(device)
    col_map = detect_columns(label_df)
    print(f'\n✅ Detected columns: {col_map}')

    if len(col_map) < 2:
        print('⚠️ Không thể tự động phát hiện đủ columns!')
        print('Columns available:', list(label_df.columns))
        print('Hãy chỉnh sửa STEP_TO_CLASS và col_map thủ công.')
        return

    # Nhóm theo video
    video_col = col_map.get('video')
    step_col = col_map.get('step')
    start_col = col_map.get('start')
    end_col = col_map.get('end')

    # Lấy danh sách video .mov (tìm cả trong thư mục con nếu có)
    video_files = {}
    for root, dirs, files in os.walk(bao_dir):
        for f in files:
            if f.lower().endswith('.mov') or f.lower().endswith('.mp4'):
                video_files[os.path.splitext(f)[0].upper()] = os.path.join(root, f)

    # Nếu không có video column → mỗi file label là 1 video
    if video_col is None:
        # Giả định: mỗi row là 1 video_file, theo thứ tự BAO1, BAO2...
        print('ℹ️ Không có cột video → dự đoán theo thứ tự file...')
        sorted_videos = sorted(video_files.keys())
        label_df['_video_'] = [sorted_videos[i] if i < len(sorted_videos)
                               else None for i in range(len(label_df))]
        video_col = '_video_'

    grouped = label_df.groupby(video_col) if video_col in label_df.columns else [(None, label_df)]

    total_saved = 0
    for vid_name, group in tqdm(grouped, desc='Processing videos'):
        vid_key = str(vid_name).upper().replace('.MOV', '').replace('.MP4', '')

        # Tìm đường dẫn video
        vid_path = video_files.get(vid_key)
        if vid_path is None:
            # Thử tìm partial match
            for k, v in video_files.items():
                if vid_key in k or k in vid_key:
                    vid_path = v
                    break

        if vid_path is None:
            print(f'⚠️ Không tìm thấy video: {vid_name}')
            continue

        print(f'\nXử lý video: {os.path.basename(vid_path)}')
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0: fps = 15.15 # Fallback fallback
        print(f'  Total frames: {total_frames}, FPS: {fps:.1f}')

        for _, row in group.iterrows():
            step_val = row[step_col] if step_col else None
            class_idx = map_step_to_class(step_val) if step_val is not None else -1

            if class_idx < 0:
                print(f'  ⚠️ Không nhận diện được step: {step_val}')
                continue

            # Xác định start/end frame
            if start_col and end_col:
                start_val = row[start_col]
                end_val = row[end_col]

                # Nếu là giây (float nhỏ) → nhân fps
                if isinstance(start_val, float) and start_val < 1000:
                    start_frame = int(start_val * fps)
                    end_frame = int(end_val * fps)
                else:
                    start_frame = int(start_val)
                    end_frame = int(end_val)
            else:
                # Không có timing → lấy toàn bộ video
                start_frame = 0
                end_frame = total_frames

            start_frame = max(0, start_frame)
            end_frame = min(end_frame, total_frames)

            if end_frame <= start_frame:
                print(f'  ⚠️ Frame range không hợp lệ: [{start_frame}, {end_frame}]')
                continue

            print(f'  Step {class_idx}: frames [{start_frame}–{end_frame}] ({end_frame-start_frame} frames)')

            img_f, skel_f = extractor.extract_frames(cap, start_frame, end_frame)

            if len(img_f) > 0:
                out_name = f'{subject_name}_{vid_key}_step{class_idx}.npy'
                out_path = os.path.join(out_dir, str(class_idx), out_name)
                np.save(out_path, {'img_feat': img_f, 'skel_feat': skel_f})
                total_saved += 1
                print(f'  ✅ Đã lưu: {out_name} (img:{img_f.shape}, skel:{skel_f.shape})')

        cap.release()

    print(f'\n🎉 HOÀN TẤT! Đã lưu {total_saved} files vào {out_dir}')


# ---- CHẠY EXTRACTION ----
if label_df is not None:
    # PREPROCESS ĐỊNH DẠNG FILE EXCEL CỦA BAO_DATA THÀNH TÍNH CHUẨN
    if 'CAMERA: 15.15 FPS' in label_df.columns:
        print("Đang tiền xử lý định dạng file Excel (Wide -> Tidy)...")
        records = []
        step_names = label_df.iloc[0, 1:7].tolist() # B1, B2, B3, B4, B5, B6

        for idx in range(1, len(label_df)):
            video_name = label_df.iloc[idx, 0]
            if pd.isna(video_name): continue

            for col_idx in range(1, 7):
                time_str = label_df.iloc[idx, col_idx]
                if pd.isna(time_str): continue
                time_str = str(time_str).strip()

                if '->' in time_str:
                    start_str, end_str = time_str.split('->')

                    def time_to_sec(t):
                        parts = t.strip().split(':')
                        if len(parts) == 2:
                            return int(parts[0]) * 60 + int(parts[1])
                        return float(t.strip())

                    try:
                        start_sec = time_to_sec(start_str)
                        end_sec = time_to_sec(end_str)
                        records.append({
                            'video': video_name,
                            'step': step_names[col_idx-1],
                            'start': start_sec,
                            'end': end_sec
                        })
                    except Exception as e:
                        print(f"Lỗi phân tích thời gian {time_str}: {e}")

        label_df_clean = pd.DataFrame(records)
        print("\nBảng dữ liệu sau khi làm sạch:")
        print(label_df_clean.head())
        extract_bao_data(label_df_clean, BAO_DIR, BAO_PROCESSED_OUT, subject_name='BAO')
    else:
        extract_bao_data(label_df, BAO_DIR, BAO_PROCESSED_OUT, subject_name='BAO')

    print('\n=== THỐNG KÊ OUTPUT ===')
    for cls in range(7):
        cls_dir = os.path.join(BAO_PROCESSED_OUT, str(cls))
        if os.path.exists(cls_dir):
            n = len(os.listdir(cls_dir))
            print(f'  Class {cls}: {n} files')
else:
    print('⚠️ Chạy Cell 4 trước để đọc label_df!')


In [ ]:
# ============================================================
# CELL 6: CHUẨN BỊ ĐƯỜNG DẪN DATASET
# ============================================================
print('='*60)
print('🔀 ĐÃ CHUYỂN SANG CHẾ ĐỘ AUTO-MERGE BẰNG DATASET CLASS')
print('='*60)
print(f"Kaggle Data Path: {KAGGLE_PROCESSED}")
print(f"Bao Data Path: {BAO_PROCESSED_OUT}")
print("\n(Thay vì copy file vật lý tốn thời gian, Dataset class ở Cell 8 sẽ tự động gộp file khi Train)")


In [ ]:
# ============================================================
# CELL 7: ĐỊNH NGHĨA MODEL (HybridTCNGRU)
# ============================================================
import torch
import torch.nn as nn

class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              padding=self.padding, dilation=dilation)

    def forward(self, x):
        out = self.conv(x)
        return out[:, :, :-self.padding] if self.padding > 0 else out


class HybridTCNGRU(nn.Module):
    def __init__(self, img_dim=576, skel_dim=126, hidden_dim=64, num_classes=7):
        super().__init__()
        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.skel_proj = nn.Sequential(
            nn.Linear(skel_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.fusion_gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
            nn.Sigmoid()
        )
        self.tcn = nn.Sequential(
            CausalConv1d(hidden_dim * 2, hidden_dim * 2, 3, dilation=1), nn.ReLU(), nn.BatchNorm1d(hidden_dim * 2),
            CausalConv1d(hidden_dim * 2, hidden_dim * 2, 3, dilation=2), nn.ReLU(), nn.BatchNorm1d(hidden_dim * 2),
            CausalConv1d(hidden_dim * 2, hidden_dim * 2, 3, dilation=4), nn.ReLU(), nn.BatchNorm1d(hidden_dim * 2),
            CausalConv1d(hidden_dim * 2, hidden_dim * 2, 3, dilation=8), nn.ReLU(), nn.BatchNorm1d(hidden_dim * 2),
        )
        self.gru = nn.GRU(hidden_dim * 2, hidden_dim, num_layers=1, batch_first=True)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, img_x, skel_x):
        h_skel = self.skel_proj(skel_x)
        h_img = self.img_proj(img_x)

        if self.training:
            mask = (torch.rand(h_img.size(0), 1, 1, device=h_img.device) > 0.75).float()
            h_img = h_img * mask
        else:
            h_img = h_img * 0.25

        cat_feat = torch.cat([h_img, h_skel], dim=-1)
        gates = self.fusion_gate(cat_feat)
        h_img_gated = h_img * gates[:, :, 0:1]
        h_skel_gated = h_skel * gates[:, :, 1:2]
        x = torch.cat([h_img_gated, h_skel_gated], dim=-1)

        x = x.transpose(1, 2)
        x = self.tcn(x)
        x = x.transpose(1, 2)

        out, _ = self.gru(x)
        out = nn.functional.dropout(out, p=0.5, training=self.training)

        attn_weights = torch.softmax(self.attention(out), dim=1)
        context = torch.sum(attn_weights * out, dim=1)

        return self.fc(context)

print('✅ Model HybridTCNGRU defined!')
model_test = HybridTCNGRU(num_classes=7)
n_params = sum(p.numel() for p in model_test.parameters())
print(f'   Parameters: {n_params:,}')

In [ ]:
# ============================================================
# CELL 8: ĐỊNH NGHĨA DATASET
# ============================================================
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import glob
import torch

class MergedHandwashDataset(Dataset):
    def __init__(self, data_dirs, seq_length=32, step_size=16,
                 subject_exclude=None, subject_include=None, is_train=True):
        self.seq_length = seq_length
        self.is_train = is_train
        self.samples = []

        if isinstance(data_dirs, str):
            data_dirs = [data_dirs]

        for data_dir in data_dirs:
            if not os.path.exists(data_dir): continue

            classes = sorted([d for d in os.listdir(data_dir)
                              if os.path.isdir(os.path.join(data_dir, d))])
            for c_name in classes:
                try: c_idx = int(c_name)
                except: continue

                files = glob.glob(os.path.join(data_dir, c_name, '*.npy'))
                for f in files:
                    basename = os.path.basename(f)

                    # Loại bỏ class 0 của Kaggle
                    if c_idx == 0 and not basename.startswith('BAO'):
                        continue

                    subject_id = self._get_subject(basename)

                    if subject_exclude and subject_id in subject_exclude:
                        continue
                    if subject_include and subject_id not in subject_include:
                        continue

                    try:
                        data = np.load(f, allow_pickle=True).item()
                        img_feat = data['img_feat'].astype(np.float32)
                        skel_feat = data['skel_feat'].astype(np.float32)
                    except:
                        continue

                    T = img_feat.shape[0]
                    if T < seq_length:
                        pad = seq_length - T
                        img_feat = np.pad(img_feat, ((0, pad), (0, 0)))
                        skel_feat = np.pad(skel_feat, ((0, pad), (0, 0)))
                        self.samples.append((img_feat, skel_feat, c_idx))
                    else:
                        for start in range(0, T - seq_length + 1, step_size):
                            self.samples.append((
                                img_feat[start:start+seq_length],
                                skel_feat[start:start+seq_length],
                                c_idx
                            ))

        if is_train:
            self._oversample()

    def _get_subject(self, basename):
        if basename.startswith('BAO'):
            parts = basename.split('_')
            if len(parts) >= 2:
                return parts[1]
            return 'BAO'
        parts = basename.replace('.npy', '').split('_')
        for i, p in enumerate(parts):
            if p == 'G' and i+1 < len(parts) and parts[i+1].isdigit():
                return f'G_{parts[i+1]}'
        return 'unknown'

    def _oversample(self):
        class_counts = {}
        for _, _, lbl in self.samples:
            class_counts[lbl] = class_counts.get(lbl, 0) + 1

        if not class_counts: return

        max_count = max(class_counts.values())
        new_samples = []
        for c, count in class_counts.items():
            c_samples = [s for s in self.samples if s[2] == c]
            if count < max_count:
                num_to_add = max_count - count
                idxs = np.random.choice(len(c_samples), num_to_add, replace=True)
                new_samples.extend([c_samples[i] for i in idxs])
        self.samples.extend(new_samples)

    def _normalize_skeleton(self, skel):
        skel = skel.copy()
        for i in range(skel.shape[0]):
            for hand_idx in range(2):
                pts = skel[i, hand_idx*21*3 : (hand_idx+1)*21*3].reshape(-1, 3)
                if np.all(pts == 0): continue
                wrist = pts[0].copy()
                pts = pts - wrist
                scale = np.max(np.linalg.norm(pts, axis=1))
                if scale > 1e-5: pts = pts / scale
                skel[i, hand_idx*21*3 : (hand_idx+1)*21*3] = pts.flatten()
        return skel

    def _augment_skeleton_3d(self, skel):
        theta = np.random.uniform(-0.1, 0.1)
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        R = np.array([
            [cos_t, -sin_t, 0],
            [sin_t, cos_t, 0],
            [0, 0, 1]
        ])
        for i in range(skel.shape[0]):
            for h in range(2):
                pts = skel[i, h*63:(h+1)*63].reshape(-1, 3)
                if np.all(pts == 0): continue
                pts = pts.dot(R.T)
                skel[i, h*63:(h+1)*63] = pts.flatten()
        return skel

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_feat, skel_feat, lbl = self.samples[idx]
        skel_feat = self._normalize_skeleton(skel_feat)
        if self.is_train:
            skel_feat = self._augment_skeleton_3d(skel_feat)
        return torch.tensor(img_feat), torch.tensor(skel_feat), torch.tensor(lbl, dtype=torch.long)

def get_bao_subjects(data_dir):
    subjects = set()
    for cls in range(7):
        cls_dir = os.path.join(data_dir, str(cls))
        if not os.path.exists(cls_dir): continue
        for f in os.listdir(cls_dir):
            if f.startswith('BAO'):
                parts = f.split('_')
                if len(parts) >= 2:
                    subjects.add(parts[1])
    def sort_key(x):
        try: return int(x.replace('BAO', ''))
        except: return x
    return sorted(list(subjects), key=sort_key)

print('✅ Đã định nghĩa MergedHandwashDataset và hàm lấy danh sách BAO subjects!')


In [ ]:
# ============================================================
# CELL 10: HÀM HUẤN LUYỆN VÀ 3 EXPERIMENTS
# ============================================================
from sklearn.metrics import classification_report
import pandas as pd
import time
import torch
import torch.nn as nn
import torch.optim as optim

def train_single_model(train_loader, val_loader, num_classes, num_epochs, device, fold_name):
    model = HybridTCNGRU(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_acc = 0.0
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    patience_limit = 10

    print(f"  ▶ Training {fold_name}...")
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        for img, skel, lbl in train_loader:
            img, skel, lbl = img.to(device), skel.to(device), lbl.to(device)
            optimizer.zero_grad()
            out = model(img, skel)
            loss = criterion(out, lbl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item() * img.size(0)
            preds = torch.argmax(out, 1)
            train_correct += (preds == lbl).sum().item()
            train_total += img.size(0)

        train_loss /= max(train_total, 1)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for img, skel, lbl in val_loader:
                img, skel, lbl = img.to(device), skel.to(device), lbl.to(device)
                out = model(img, skel)
                loss = criterion(out, lbl)
                val_loss += loss.item() * img.size(0)
                preds = torch.argmax(out, 1)
                val_correct += (preds == lbl).sum().item()
                val_total += img.size(0)

        val_loss /= max(val_total, 1)
        val_acc = val_correct / max(val_total, 1) * 100

        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
            star = "⭐"
        else:
            patience_counter += 1
            star = f"(patience {patience_counter}/{patience_limit})"

        print(f"    Epoch {epoch+1:02d}/{num_epochs} | Loss {train_loss:.4f}/{val_loss:.4f} | Acc {val_acc:.2f}% {star}")

        if patience_counter >= patience_limit:
            print(f"    🛑 Early stop at epoch {epoch+1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, best_val_loss, best_val_acc

from sklearn.metrics import classification_report
import pandas as pd
import time

bao_subjects = get_bao_subjects(BAO_PROCESSED_OUT)
print(f'🚀 Tìm thấy {len(bao_subjects)} BAO Subjects: {bao_subjects}')

BATCH_SIZE = 128
NUM_EPOCHS = 40
num_classes = 7
results_exp1 = []
results_exp2 = []
results_exp3 = []

def eval_model_on_subject(model, val_subject):
    val_ds = MergedHandwashDataset([BAO_PROCESSED_OUT], subject_include=[val_subject], is_train=False)
    if len(val_ds) == 0: return 0.0
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    correct = total = 0
    model.eval()
    with torch.no_grad():
        for img, skel, lbl in val_loader:
            img, skel, lbl = img.to(device), skel.to(device), lbl.to(device)
            out = model(img, skel)
            preds = torch.argmax(out, 1)
            correct += (preds == lbl).sum().item()
            total += lbl.size(0)
    return correct / max(total, 1) * 100

# ------------------------------------------------------------
# EXPERIMENT 1: BAO ONLY (Train: BAO, Test: BAO)
# ------------------------------------------------------------
print('\n' + '='*60)
print('🔥 KỊCH BẢN 1: BAO ONLY (LOSO trên tập dữ liệu BAO)')
print('='*60)
for fold_idx, val_subject in enumerate(bao_subjects):
    print(f'\n--- EXP 1 | FOLD {fold_idx+1}/{len(bao_subjects)} | Val Subject: {val_subject} ---')
    train_ds = MergedHandwashDataset([BAO_PROCESSED_OUT], subject_exclude=[val_subject], is_train=True)
    val_ds = MergedHandwashDataset([BAO_PROCESSED_OUT], subject_include=[val_subject], is_train=False)

    if len(train_ds) == 0 or len(val_ds) == 0: continue
    print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

    model, _, best_acc = train_single_model(
        train_loader, val_loader, num_classes, NUM_EPOCHS, device, f'EXP1_{val_subject}')
    results_exp1.append({'subject': val_subject, 'acc': best_acc})

# ------------------------------------------------------------
# EXPERIMENT 2: KAGGLE ONLY (Train 1 lần trên toàn bộ Kaggle, Test BAO)
# ------------------------------------------------------------
print('\n' + '='*60)
print('🔥 KỊCH BẢN 2: KAGGLE ONLY (Train Toàn bộ Kaggle, Test trên từng video BAO)')
print('='*60)
print('▶ Bước 1: Huấn luyện 1 Model duy nhất bằng TOÀN BỘ dữ liệu Kaggle...')
train_ds_kaggle = MergedHandwashDataset([KAGGLE_PROCESSED], is_train=True)
train_loader_k = DataLoader(train_ds_kaggle, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

if len(train_ds_kaggle) > 0:
    # Cần 1 dummy val_loader cho hàm train_single_model chạy
    dummy_val_ds = MergedHandwashDataset([BAO_PROCESSED_OUT], subject_include=[bao_subjects[0]], is_train=False)
    dummy_val_loader = DataLoader(dummy_val_ds, batch_size=BATCH_SIZE, shuffle=False)

    kaggle_model, _, _ = train_single_model(
        train_loader_k, dummy_val_loader, num_classes, NUM_EPOCHS, device, 'EXP2_Kaggle_All')

    print('\n▶ Bước 2: Đánh giá Model Kaggle này trên từng BAO subject (Test set thay đổi 14 lần)...')
    for fold_idx, val_subject in enumerate(bao_subjects):
        acc = eval_model_on_subject(kaggle_model, val_subject)
        print(f'   + Test {val_subject} (Fold {fold_idx+1}): {acc:.2f}%')
        results_exp2.append({'subject': val_subject, 'acc': acc})
else:
    print('❌ Không tìm thấy dữ liệu Kaggle để train kịch bản 2!')

# ------------------------------------------------------------
# EXPERIMENT 3: KAGGLE + BAO (Gộp Chung - Train: Kaggle+13 BAO, Test: 1 BAO)
# ------------------------------------------------------------
print('\n' + '='*60)
print('🔥 KỊCH BẢN 3: KAGGLE + BAO (LOSO trên tập BAO + Data Kaggle)')
print('='*60)
for fold_idx, val_subject in enumerate(bao_subjects):
    print(f'\n--- EXP 3 | FOLD {fold_idx+1}/{len(bao_subjects)} | Val Subject: {val_subject} ---')
    train_ds = MergedHandwashDataset([KAGGLE_PROCESSED, BAO_PROCESSED_OUT], subject_exclude=[val_subject], is_train=True)
    val_ds = MergedHandwashDataset([BAO_PROCESSED_OUT], subject_include=[val_subject], is_train=False)

    if len(train_ds) == 0 or len(val_ds) == 0: continue
    print(f'Train: {len(train_ds)} samples (Kaggle + BAO) | Val: {len(val_ds)} samples (Chỉ BAO)')

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

    model, _, best_acc = train_single_model(
        train_loader, val_loader, num_classes, NUM_EPOCHS, device, f'EXP3_{val_subject}')
    results_exp3.append({'subject': val_subject, 'acc': best_acc})

# ============================================================
# TỔNG KẾT BÁO CÁO 3 KỊCH BẢN
# ============================================================
print('\n' + '='*80)
print('🏆 BẢNG TỔNG KẾT SO SÁNH 3 KỊCH BẢN (ACCURACY %)')
print('='*80)

df_res = pd.DataFrame({'Subject (Val)': bao_subjects})
df_res['Exp 1 (BAO Only)'] = [next((r['acc'] for r in results_exp1 if r['subject'] == s), 0) for s in bao_subjects]
df_res['Exp 2 (Kaggle Only)'] = [next((r['acc'] for r in results_exp2 if r['subject'] == s), 0) for s in bao_subjects]
df_res['Exp 3 (Kaggle + BAO)'] = [next((r['acc'] for r in results_exp3 if r['subject'] == s), 0) for s in bao_subjects]

print(df_res.to_string(index=False, float_format="%.2f"))

print('\n📊 TRUNG BÌNH TOÀN BỘ (MEAN ACCURACY):')
print(f"   ▶ Kịch bản 1 (BAO Only)   : {df_res['Exp 1 (BAO Only)'].mean():.2f}%")
print(f"   ▶ Kịch bản 2 (Kaggle Only): {df_res['Exp 2 (Kaggle Only)'].mean():.2f}%")
print(f"   ▶ Kịch bản 3 (Kaggle+BAO) : {df_res['Exp 3 (Kaggle + BAO)'].mean():.2f}%")
print('='*80)


In [ ]:
# Cũ: Hàm LOSO_RUN đã được chuyển lên trên Cell 10.


In [ ]:
# ============================================================
# CELL 11: TRAIN FINAL MODEL TRÊN TOÀN BỘ DATA
# (Dùng để deploy vào app_main.py)
# ============================================================
print('🚀 Training final model trên toàn bộ data...')

full_ds = MergedHandwashDataset(MERGED_DIR, is_train=True)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2 if torch.cuda.is_available() else 0,
                         pin_memory=True)

print(f'  Total training samples: {len(full_ds)}')
from collections import Counter
dist = Counter(s[2] for s in full_ds.samples)
print(f'  Distribution: {dict(sorted(dist.items()))}')

final_model = HybridTCNGRU(num_classes=7).to(device)

class_counts = np.zeros(7)
for _, _, lbl in full_ds.samples:
    class_counts[lbl] += 1
weights = np.ones(7)
valid = class_counts > 0
weights[valid] = 1.0 / class_counts[valid]
weights = weights / weights.sum() * 7
class_w = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = FocalLoss(weight=class_w, gamma=2.0, label_smoothing=0.1)
optimizer = torch.optim.Adam(final_model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

FINAL_EPOCHS = NUM_EPOCHS + 10  # Thêm epochs vì không cần val
for epoch in range(FINAL_EPOCHS):
    final_model.train()
    loss_total = 0.0
    correct = total = 0
    for img, skel, lbl in full_loader:
        img, skel, lbl = img.to(device), skel.to(device), lbl.to(device)
        optimizer.zero_grad()
        out = final_model(img, skel)
        loss = criterion(out, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
        optimizer.step()
        loss_total += loss.item()
        correct += (torch.argmax(out, 1) == lbl).sum().item()
        total += lbl.size(0)
    scheduler.step()
    acc = correct / max(total, 1) * 100
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:03d}/{FINAL_EPOCHS} | Loss: {loss_total/len(full_loader):.4f} | Train Acc: {acc:.1f}%')

# Lưu final model
torch.save(final_model.state_dict(), f'{MODELS_DIR}/final_model_all_data.pth')
print(f'\n✅ Final model saved → {MODELS_DIR}/final_model_all_data.pth')

In [ ]:
# ============================================================
# CELL 12: LƯU KẾT QUẢ VỀ GOOGLE DRIVE
# ============================================================
import shutil, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
drive_out = f'/content/drive/MyDrive/Webcam_Handwash/training_results_{timestamp}'
os.makedirs(drive_out, exist_ok=True)

# Copy results
shutil.copytree(RESULTS_DIR, os.path.join(drive_out, 'results'), dirs_exist_ok=True)
shutil.copytree(MODELS_DIR, os.path.join(drive_out, 'models'), dirs_exist_ok=True)

# Tóm tắt
summary_text = f'Training Summary ({timestamp})\n'
summary_text += f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}\n'
summary_text += f'Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE}\n\n'
summary_text += 'LOSO Results:\n'
for r in fold_results:
    summary_text += f"  {r['subject']}: {r['final_acc']:.1f}%\n"
if fold_results:
    summary_text += f"  Mean: {np.mean([r['final_acc'] for r in fold_results]):.1f}%"

with open(os.path.join(drive_out, 'summary.txt'), 'w') as f_:
    f_.write(summary_text)

print(f'✅ Đã lưu tất cả về Drive: {drive_out}')
print(summary_text)